# Boat Vision — combined maritime model (free GPU)

Trains ONE model that detects **boats, buoys/seamarks, persons/swimmers, floating objects, and kayaks** by merging three datasets:
Singapore Maritime (boats/buoys/kayaks), Swimmer (people in water), Marine debris (floating objects).

**Steps:**
1. **Runtime → Change runtime type → T4 GPU → Save**.
2. **Runtime → Run all**, click **Run anyway** on the warning.
3. Paste your Roboflow API key when asked.
4. Wait (~1–2 h). At the end it downloads **best.pt**.
5. Put **best.pt** into Boat Vision at `models\maritime\best.pt` and restart the app.

In [ ]:
import torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(set Runtime->GPU)')
!pip -q install ultralytics roboflow pyyaml

In [ ]:
import getpass
from roboflow import Roboflow
rf = Roboflow(api_key=getpass.getpass('Paste your Roboflow API key and press Enter: '))
sources = []
sources.append(('mar',  rf.workspace('maritime-cumkb').project('singapore-maritime').version(5).download('yolov8').location))
sources.append(('swim', rf.workspace('maritime-cumkb').project('swimmer--swimmer').version(2).download('yolov8').location))
sources.append(('deb',  rf.workspace('tamkang').project('marine-debris-yolo').version(1).download('yolov8').location))
print('downloaded:', sources)

In [ ]:
import os, glob, shutil, yaml
MERGED = '/content/merged'
UNIFIED = ['boat', 'navigation_buoy', 'person', 'floating_object', 'kayak']
U = {n: i for i, n in enumerate(UNIFIED)}

def to_unified(name):
    n = name.strip().lower()
    if n in ('kayak',): return U['kayak']
    if n in ('boat','ferry','sail boat','sailboat','speed boat','speedboat','vessel-ship','vessel','ship','jetski'): return U['boat']
    if n in ('buoy','navigation buoy','navigation_buoy','navigation mark','sea mark','seamark'): return U['navigation_buoy']
    if n in ('human','person','swimmer','in_water','people','floater'): return U['person']
    if n in ('other','floating object','floating_object','debris','can','foam','plastic bottle','plastic','unknow','unknown','trash'): return U['floating_object']
    return None

for s in ('train','valid'):
    os.makedirs(f'{MERGED}/{s}/images', exist_ok=True)
    os.makedirs(f'{MERGED}/{s}/labels', exist_ok=True)

def process(loc, tag):
    names = yaml.safe_load(open(os.path.join(loc, 'data.yaml')))['names']
    if isinstance(names, dict): names = [names[k] for k in sorted(names)]
    idxmap = {i: to_unified(n) for i, n in enumerate(names)}
    for split in ('train', 'valid', 'test'):
        img_dir = os.path.join(loc, split, 'images')
        lbl_dir = os.path.join(loc, split, 'labels')
        if not os.path.isdir(img_dir): continue
        dst = 'valid' if split == 'valid' else 'train'
        for img in glob.glob(os.path.join(img_dir, '*')):
            base, ext = os.path.splitext(os.path.basename(img))
            lbl = os.path.join(lbl_dir, base + '.txt')
            lines = []
            if os.path.exists(lbl):
                for line in open(lbl):
                    p = line.split()
                    if not p: continue
                    ui = idxmap.get(int(float(p[0])))
                    if ui is None: continue
                    lines.append(' '.join([str(ui)] + p[1:]))
            out = f'{tag}_{base}'
            shutil.copy(img, f'{MERGED}/{dst}/images/{out}{ext}')
            open(f'{MERGED}/{dst}/labels/{out}.txt', 'w').write('\n'.join(lines))

for tag, loc in sources:
    process(loc, tag)

open(f'{MERGED}/data.yaml', 'w').write(yaml.safe_dump(
    {'path': MERGED, 'train': 'train/images', 'val': 'valid/images', 'nc': len(UNIFIED), 'names': UNIFIED}))
for s in ('train','valid'):
    print(s, len(glob.glob(f'{MERGED}/{s}/images/*')), 'images')
print('classes:', UNIFIED)

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo26s.pt')
model.train(data=f'{MERGED}/data.yaml', epochs=40, imgsz=640, batch=16, name='maritime_combined')

In [ ]:
best = str(model.trainer.best)
print('Best weights:', best)
from google.colab import files
files.download(best)